<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🔀 Lab 03b: Build a Multi-Agent Travel Workflow</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 ここで学ぶこと

このラボでは、複数の専門エージェントを **マルチエージェントワークフロー** にオーケストレーションする方法を学びます。`WorkflowAgentDefinition` を使用して、3つのドメイン特化型エージェント — **Flight Agent**、**Hotel Agent**、**Car Rental Agent** — を作成し、YAMLベースのワークフローで顧客のクエリを適切な専門家にルーティングします。

> **これは専門家チームのようなものです。** すべてを少しずつ知っている一般的なエージェントの代わりに、特定の分野に特化した専門家 — フライトアドバイザー、ホテルコンシェルジュ、レンタカーデスク — を集めます。ワークフローは、各顧客のリクエストを適切な専門家にルーティングし、その回答を統合するディスパッチャーの役割を果たします。

---


## 🧳 Contoso Travel ストーリー

Lab 03a が開発したContoso Travelのエージェントは、実際にフライト、ホテル、レンタカーを検索できるという点では機能していました。しかし、チームが機能を追加するにつれて、問題が発生し始めました。フライト予約のルール、ホテルのアメニティの詳細、レンタカーのポリシー、一般的な旅行アドバイスなど、すべてを一度に網羅しようとした結果、システムプロンプトは2,000語以上に膨れ上がりました。テスト中に、ある顧客が「ニューヨークからロンドンへの旅行を計画してください。フライト、劇場街近くの素敵なホテル、日帰り旅行用のレンタカーが必要です」と質問しました。エージェントはフライトを検索しましたが、ホテルのリクエストを忘れてしまったようでした。指摘されると、ホテルは検索しましたが、ホテルの説明の中にレンタカーのアドバイスが混ざって表示されました。**このエージェントはあらゆる分野の専門家になろうとして、細部への対応が不十分だったのです。**

### 🔴 現在直面している問題

- **1つのエージェントではすべてをうまくこなせない。** ドメイン（フライト、ホテル、レンタカー）や複雑なロジックを追加すると、単一のエージェントの指示は膨れ上がり、矛盾が生じます。
- フォーカスを失い、ツール呼び出しが混乱し、一貫性のない結果を生み出します。
- 実際の旅行代理店には専門のアドバイザーがいる理由があります — AIシステムにも同じアーキテクチャが必要です。
- では、1つのエージェントを複数の専門家に分割し、どのように調整するのでしょうか？

### ✅ このラボで解決すること

このラボでは、`WorkflowAgentDefinition` を使用した **マルチエージェントワークフロー** を紹介します:
 - 3つの専門エージェント — フライトエージェント、ホテルエージェント、レンタカーエージェント — を作成し、それぞれに特化した指示とドメイン固有のツールを持たせます。
 - YAMLベースのワークフローでそれらを接続します。
 - 顧客のクエリを適切な専門家にルーティングし、その回答を統合して統一された旅行プランを作成します。

## 2. Setup

---


In [ ]:
# セットアップ: インポートと Microsoft Foundry クライアント構成
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    # YAMLワークフローを介して複数のエージェントをオーケストレーション
    WorkflowAgentDefinition,
    FunctionTool,
    Tool,
)

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
# allow_preview=True is REQUIRED for WorkflowAgentDefinition
project_client = AIProjectClient(endpoint=endpoint, credential=credential, allow_preview=True)
openai_client = project_client.get_openai_client()

print(f"✅ Microsoft Foundry に接続されました (プレビュー機能が有効化されています)")

## 3. 専門エージェントの作成

3つの専門エージェントを作成します — 各エージェントは1つの旅行ドメインの専門家です — さらに、彼らの結果を統合して統一された旅行プランを作成する **コンシェルジュエージェント (Concierge Agent)** も作成します。

```
Customer Query
  ├─→ ✈️ Flight Agent     → flight options
  ├─→ 🏨 Hotel Agent      → hotel options
  ├─→ 🚗 Car Agent        → car rental options
  └─→ 🎩 Concierge Agent  → combines all into final trip plan
```

---


In [ ]:
# 4つの PromptAgentDefinitions を作成します。
# 3つの専門エージェント + 1つのコンシェルジュエージェント (結果を統合)
# create_version() はエージェントをサーバー側で登録し、
# 参照用に .name と .version を持つオブジェクトを返します。

# Flight Specialist Agent
flight_agent = project_client.agents.create_version(
    agent_name="contoso-flight-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="""あなたはコントソ・トラベルのフライトスペシャリストです。
お客様のフライト探しをお手伝いします。旅行のリクエストを受けたら、以下の手順に従ってください。
1. 出発地と目的地を特定します。
2. おおよその料金とともに、フライトオプションを提案します。
3. ご希望の座席クラスが指定されていない場合は、お伺いします。
4. 簡潔に、箇条書きで分かりやすく回答してください。

回答の最後には必ず [FLIGHT SEARCH COMPLETE] と入力してください。""",
    ),
)
print(f"✅ Flight Agent created: {flight_agent.name} (v{flight_agent.version})")

# Hotel Specialist Agent
hotel_agent = project_client.agents.create_version(
    agent_name="contoso-hotel-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="""あなたはコントソ・トラベルのホテルスペシャリストです。
お客様のホテル探しをお手伝いします。旅行のリクエストを受けたら、以下の手順に従ってください。
1. 目的地の都市を特定します。
2. 異なる星評価と価格帯のホテルオプションを提案します。
3. 主要なアメニティを強調します。
4. 簡潔に、箇条書きで分かりやすく回答してください。

回答の最後には必ず [HOTEL SEARCH COMPLETE] と入力してください。""",
    ),
)
print(f"✅ Hotel Agent created: {hotel_agent.name} (v{hotel_agent.version})")

# Car Rental Specialist Agent
car_agent = project_client.agents.create_version(
    agent_name="contoso-car-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="""あなたはコントソ・トラベルのレンタカー専門家です。
お客様のレンタカー探しをお手伝いします。旅行のリクエストを受けたら、以下の手順に従ってください。
1. レンタカーが必要な都市を特定します。
2. 車種と料金を提案します。
3. 利用可能性とピックアップの詳細を記載します。
4. 簡潔に、箇条書きで分かりやすく回答してください。

回答の最後には必ず [CAR RENTAL SEARCH COMPLETE] と入力してください。""",
    ),
)
print(f"✅ Car Rental Agent created: {car_agent.name} (v{car_agent.version})")

# Concierge Agent — combines all specialist responses
concierge_agent = project_client.agents.create_version(
    agent_name="contoso-concierge-agent",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="""あなたはコントソ・トラベルのコンシェルジュです。
3人の専門エージェント（フライト、ホテル、レンタカー）からの調査結果を受け取り、それらを統合して顧客向けの整理された旅行プランを作成します。

回答の形式は以下の通りです:
## ✈️ フライト
(フライトオプションの要約)

## 🏨 ホテル
(ホテルオプションの要約)

## 🚗 レンタカー
(レンタカーオプションの要約)

## 📋 旅行の概要
(総費用の見積もり範囲を含む簡単な概要)

簡潔でありながら包括的に。最適なオプションを明確に提示してください。""",
    ),
)
print(f"✅ Concierge Agent created: {concierge_agent.name} (v{concierge_agent.version})")

## 4. ワークフロー YAML の定義

ワークフローは、顧客のリクエストを各専門エージェントにルーティングし、3つの応答をすべて **コンシェルジュエージェント (Concierge Agent)** に渡して統合された旅行プランを作成します。

---


In [ ]:
# エージェントをオーケストレーションするYAMLベースのワークフローを定義します。
# このワークフローは、各スペシャリストごとに独立した会話を作成し、
# それらを順番に呼び出し、応答を変数に格納します。
#
# 注: SendActivity は `activity` をリテラルテキストとして扱います。式は評価しません。
# 実際のエージェント応答テキストを取得するには、
# ワークフロー完了後にクライアント側でサブ会話メッセージを取得します。

workflow_yaml = f"""
kind: workflow
trigger:
  kind: OnConversationStart
  id: contoso_travel_workflow
  actions:
    - kind: SetVariable
      id: capture_user_request
      variable: Local.LatestMessage
      value: "=UserMessage(System.LastMessageText)"

    - kind: CreateConversation
      id: create_flight_conversation
      conversationId: Local.FlightConversationId

    - kind: CreateConversation
      id: create_hotel_conversation
      conversationId: Local.HotelConversationId

    - kind: CreateConversation
      id: create_car_conversation
      conversationId: Local.CarConversationId

    - kind: InvokeAzureAgent
      id: invoke_flight_agent
      description: Search for flights
      conversationId: "=Local.FlightConversationId"
      agent:
        name: {flight_agent.name}
      input:
        messages: "=Local.LatestMessage"
      output:
        messages: Local.FlightResponse

    - kind: InvokeAzureAgent
      id: invoke_hotel_agent
      description: Search for hotels
      conversationId: "=Local.HotelConversationId"
      agent:
        name: {hotel_agent.name}
      input:
        messages: "=Local.LatestMessage"
      output:
        messages: Local.HotelResponse

    - kind: InvokeAzureAgent
      id: invoke_car_agent
      description: Search for car rentals
      conversationId: "=Local.CarConversationId"
      agent:
        name: {car_agent.name}
      input:
        messages: "=Local.LatestMessage"
      output:
        messages: Local.CarResponse

    - kind: EndConversation
      id: end_workflow
"""

print("✅ Workflow YAML defined")
print(f"   Specialists: {flight_agent.name}, {hotel_agent.name}, {car_agent.name}")
print(f"   Concierge: {concierge_agent.name} (called client-side after workflow)")

## 5. ワークフローエージェントを作成する

Microsoft Foundry にワークフローを登録します。

---


In [ ]:
# WorkflowAgentDefinition は、指示ではなく YAML を受け取ります。
# PromptAgentDefinition (単一の LLM + プロンプト) とは異なり、
# ワークフローエージェントはアクションを介して複数のエージェントをオーケストレーションします。
workflow = project_client.agents.create_version(
    agent_name="contoso-travel-workflow",
    definition=WorkflowAgentDefinition(workflow=workflow_yaml),
)

print(f"✅ ワークフローエージェントが作成されました！")
print(f"   名前: {workflow.name}")
print(f"   バージョン: {workflow.version}")
print(f"   ID: {workflow.id}")

## 6. テスト: エンドツーエンドの旅行計画

旅行計画のリクエストを送信し、ワークフローが3つの専門エージェントをどのようにオーケストレーションするかを確認します。

---


In [ ]:
# ワークフローを OpenAI 互換の Responses API 経由で実行する
conversation = openai_client.conversations.create()
print(f"✅ 会話が作成されました (ID: {conversation.id})")

user_input = "ニューヨークからロンドンへの旅行を計画してください。フライト、ホテル、レンタカーが必要です。"

print("\n┌" + "─" * 70 + "┐")
print("│ 📨 初期クエリ" + " " * 53 + "│")
print("├" + "─" * 70 + "┤")
for line in [user_input[i:i+68] for i in range(0, len(user_input), 68)]:
    print(f"│ {line:<68} │")
print("└" + "─" * 70 + "┘")

stream = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": workflow.name, "type": "agent_reference"}},
    input=user_input,
    stream=True,
)

response_id = None
actions_log = []

print("\n⏳ ワークフロー実行中...\n")
for event in stream:
    if event.type == "response.output_item.added" and event.item.type == "workflow_action":
        print(f"  🔄 '{event.item.action_id}' 開始")
    elif event.type == "response.output_item.done" and event.item.type == "workflow_action":
        print(f"  ✅ '{event.item.action_id}' 完了")
        actions_log.append(event.item.action_id)
    elif event.type == "response.completed":
        response_id = event.response.id

print(f"\n✅ ワークフロー完了! ({len(actions_log)} アクション)")
print(f"   レスポンス ID: {response_id}")

## 7. エージェントの応答を取得して結合する

ワークフローが専門エージェントをオーケストレーションした後、会話から実際の応答を取得し、それらを **コンシェルジュエージェント (Concierge Agent)** に渡して、統合された旅行プランを作成します。

---


In [ ]:
# ワークフロー完了後、専門家による実際の回答を取得します。
# ワークフローは回答をサブ会話に保存します。
# 会話アイテムAPIを介してアクセスできます。
response = openai_client.responses.retrieve(response_id=response_id)

# ワークフローアクションアイテムからサブ会話IDを収集
sub_conv_ids = []
for item in response.output:
    if item.type == "workflow_action":
        raw = item.model_dump()
        # CreateConversation アクションは会話IDを保存する場合があります
        for key, val in raw.items():
            if isinstance(val, str) and val.startswith("conv_"):
                sub_conv_ids.append((item.action_id, val))

# メイン会話からアイテムをリスト化してみる
print("┌" + "─" * 70 + "┐")
print("│ 🔍 エージェントの応答を取得中" + " " * 40 + "│")
print("├" + "─" * 70 + "┤")

specialist_texts = {}
agent_labels = {"flight": "✈️  フライト", "hotel": "🏨 ホテル", "car": "🚗 レンタカー"}

try:
    items = openai_client.conversations.items.list(conversation_id=conversation.id)
    for ci in items.data:
        raw = ci.model_dump()
        role = raw.get("role", "")
        if role == "assistant":
            content = raw.get("content", [])
            if content and isinstance(content, list):
                for c in content:
                    text = c.get("text", "")
                    if text and len(text) > 20:
                        # Identify which agent by checking markers
                        for key, label in agent_labels.items():
                            if key.upper() in text.upper():
                                specialist_texts[key] = (label, text)
                                break
                        else:
                            specialist_texts[f"other_{len(specialist_texts)}"] = ("🤖 エージェント", text)
except Exception:
    pass

# If conversation items didn't yield results, try sub-conversations
if not specialist_texts and sub_conv_ids:
    for action_id, conv_id in sub_conv_ids:
        try:
            items = openai_client.conversations.items.list(conversation_id=conv_id)
            for ci in items.data:
                raw = ci.model_dump()
                if raw.get("role") == "assistant":
                    content = raw.get("content", [])
                    if content and isinstance(content, list):
                        text = content[0].get("text", "")
                        if text:
                            specialist_texts[action_id] = (action_id, text)
        except Exception:
            pass

# Display what we found
if specialist_texts:
    for key in ["flight", "hotel", "car"]:
        if key in specialist_texts:
            label, text = specialist_texts[key]
            print(f"│ {label} エージェントの応答:" + " " * (54 - len(label)) + "│")
            print("│" + "─" * 70 + "│")
            for line in text.split('\n'):
                while len(line) > 68:
                    print(f"│ {line[:68]} │")
                    line = line[68:]
                print(f"│ {line:<68} │")
            print("│" + " " * 70 + "│")
    # Show any other texts we found
    for key, (label, text) in specialist_texts.items():
        if key not in ["flight", "hotel", "car"]:
            print(f"│ {label} エージェントの応答:" + " " * (60 - len(label)) + "│")
            print("│" + "─" * 70 + "│")
            for line in text.split('\n'):
                while len(line) > 68:
                    print(f"│ {line[:68]} │")
                    line = line[68:]
                print(f"│ {line:<68} │")
            print("│" + " " * 70 + "│")
else:
    print("│ (専門家の応答はワークフロー内部のコンテキストに保存されています)" + " " * 12 + "│")
    print("│ コンシェルジュエージェントを直接呼び出して統合された旅行プランを作成します..." + " " * 12 + "│")

print("└" + "─" * 70 + "┘")

# コンシェルジュエージェントを呼び出して統合された旅行プランを作成
print("\n🎩 コンシェルジュエージェントを呼び出して統合された旅行プランを作成中...\n")

concierge_input = user_input
if specialist_texts:
    parts = []
    for key in ["flight", "hotel", "car"]:
        if key in specialist_texts:
            label, text = specialist_texts[key]
            parts.append(f"--- {label} エージェント ---\n{text}")
    concierge_input = (
        f"顧客のリクエスト: {user_input}\n\n"
        "以下は専門家エージェントの応答です:\n\n" +
        "\n\n".join(parts) +
        "\n\nこれらを統合して統一された旅行プランを作成してください。"
    )

concierge_conv = openai_client.conversations.create()
concierge_stream = openai_client.responses.create(
    conversation=concierge_conv.id,
    extra_body={"agent_reference": {"name": concierge_agent.name, "type": "agent_reference"}},
    input=concierge_input,
    stream=True,
)

concierge_response_id = None
for event in concierge_stream:
    if event.type == "response.completed":
        concierge_response_id = event.response.id

concierge_response = openai_client.responses.retrieve(response_id=concierge_response_id)
final_text = ""
for item in concierge_response.output:
    if item.type == "message" and item.content:
        final_text = item.content[0].text
        break

print("┌" + "─" * 70 + "┐")
print("│ 🎩 統合された旅行プラン (コンシェルジュエージェント)" + " " * 18 + "│")
print("├" + "─" * 70 + "┤")
for line in final_text.split('\n'):
    while len(line) > 68:
        print(f"│ {line[:68]} │")
        line = line[68:]
    print(f"│ {line:<68} │")
print("└" + "─" * 70 + "┘")

# コンシェルジュ会話のクリーンアップ
openai_client.conversations.delete(conversation_id=concierge_conv.id)

## 8. Cleanup

---


In [ ]:
# クリーンアップ: リモートリソースを逆順に削除
openai_client.conversations.delete(conversation_id=conversation.id)
print("✅ 会話を削除しました")

# ワークフローを先に削除し、その後に参照しているエージェントを削除
project_client.agents.delete_version(agent_name=workflow.name, agent_version=workflow.version)
print("✅ ワークフローを削除しました")

project_client.agents.delete_version(agent_name=concierge_agent.name, agent_version=concierge_agent.version)
print("✅ コンシェルジュエージェントを削除しました")

project_client.agents.delete_version(agent_name=flight_agent.name, agent_version=flight_agent.version)
print("✅ フライトエージェントを削除しました")

project_client.agents.delete_version(agent_name=hotel_agent.name, agent_version=hotel_agent.version)
print("✅ ホテルエージェントを削除しました")

project_client.agents.delete_version(agent_name=car_agent.name, agent_version=car_agent.version)
print("✅ カーレンタルエージェントを削除しました")

openai_client.close()
project_client.close()
credential.close()
print("✅ クライアントを閉じました")

## 9. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ 達成したこと</h3>
  <ul style="margin-bottom: 0;">
  <li>3つの専門エージェント（フライト、ホテル、カーレンタル）を作成し、それぞれにドメイン固有の指示を設定</li>
  <li>3つのエージェントをオーケストレーションするYAMLベースのワークフローを定義</li>
  <li>ストリーミングワークフローイベントを使用してエンドツーエンドの旅行計画をテスト</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>💡 重要なポイント:</strong> マルチエージェントワークフローを使用すると、複雑なタスクを専門エージェントに分解できます。各エージェントは自分の得意分野に集中し、ワークフローがオーケストレーションを担当します。これが本番環境向けのエージェントシステムの構築方法です。
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ 次のステップ:</strong> <a href="lab-04-tracing.ipynb">Lab 04</a>では、<b>OpenTelemetryトレーシング</b>を追加して、ユーザーのクエリからツール呼び出し、最終的な応答まで、旅行エージェント内で何が起こっているかを正確に観察します。
</div>

---
